<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/07_hpc_scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 7 of 7: Scaling from Laptop to HPC

*OpenImpala Tutorial Series — From first solve to HPC deployment*

---

The previous tutorials all ran on a single machine. Real synchrotron datasets, however, can be $2000^3$ voxels or larger — too big to fit in a single node's memory, let alone solve interactively.

OpenImpala is built on AMReX and MPI, so the same Python script you wrote in earlier tutorials can be launched with `mpirun` across many nodes. For very large files, OpenImpala's C++ readers can also bypass Python entirely, reading TIFF and HDF5 data in parallel directly from disk.

This tutorial is primarily a **reference guide**. The code cells are designed to be saved as standalone scripts and submitted to an HPC scheduler, rather than run inside this notebook.

**What you will learn:**
1. How OpenImpala distributes work across MPI ranks.
2. The role of `max_grid_size` in domain decomposition.
3. A complete MPI-ready Python script.
4. How to use the C++ parallel readers for out-of-core I/O.
5. A ready-to-use SLURM batch submission script.

**Prerequisites:** Familiarity with the OpenImpala Python API ([Tutorials 1–2](01_hello_openimpala.ipynb)) and access to a multi-core machine or HPC cluster. All earlier tutorials in the series can inform what scripts you deploy at scale.

## 1. How OpenImpala Distributes Work

When you call `oi.tortuosity(data, ...)`, here's what happens under the hood:

1. **Domain Decomposition:** The 3D voxel grid is split into boxes of size `max_grid_size^3`. AMReX distributes these boxes across MPI ranks.
2. **HYPRE Setup:** Each rank fills its local portion of the HYPRE matrix (7-point stencil, harmonic mean face coefficients).
3. **Parallel Solve:** HYPRE solves the global system using its own MPI-aware algorithms (FlexGMRES, PCG, etc.).
4. **Flux Integration:** Each rank computes its local flux contribution, then MPI reduces to a global sum.

The key tuning parameter is **`max_grid_size`**:
- Too small (e.g., 16): many tiny boxes = high MPI communication overhead.
- Too large (e.g., 512): few huge boxes = poor load balancing, high memory per rank.
- Sweet spot: typically **32--128** for most workloads.

```
Domain: 256^3 voxels, max_grid_size=64
-> 64 boxes of 64^3 each
-> On 16 MPI ranks: 4 boxes per rank
-> On 64 MPI ranks: 1 box per rank (ideal)
```

## 2. An MPI-Ready Python Script

The script below is a complete standalone Python script that can be launched with MPI. The key point: you do not need to change any OpenImpala API calls when going from 1 core to 128. MPI parallelism is handled internally by AMReX and HYPRE.

In [ ]:
# === hpc_tortuosity.py ===
# Run with: mpirun -np 128 python hpc_tortuosity.py
#
# This script is identical to what you would run on a laptop,
# but MPI distributes the work automatically.

SCRIPT = '''
import time
import numpy as np
import openimpala as oi

# For large files, use OpenImpala's built-in parallel readers
# instead of loading everything into NumPy first.
IMAGE_PATH = "/path/to/synchrotron_scan.hdf5"
HDF5_DATASET = "/exchange/data"
PHASE_ID = 1
MAX_GRID_SIZE = 64  # Tune this for your cluster

with oi.Session():
    t0 = time.time()

    # --- Option A: NumPy workflow (for datasets that fit in RAM) ---
    # import tifffile
    # data = tifffile.imread("my_scan.tif").astype(np.int32)
    # vf = oi.volume_fraction(data, phase=PHASE_ID)
    # tau = oi.tortuosity(data, phase=PHASE_ID, direction="z",
    #                     max_grid_size=MAX_GRID_SIZE)

    # --- Option B: C++ parallel reader (for massive datasets) ---
    # The C++ reader distributes the file read across MPI ranks,
    # so no single node needs to hold the full dataset in memory.
    reader, img = oi.read_image(
        IMAGE_PATH,
        threshold=0.5,
        file_format="hdf5",
        hdf5_dataset=HDF5_DATASET,
        max_grid_size=MAX_GRID_SIZE,
    )

    # Volume fraction (works on the distributed VoxelImage directly)
    vf = oi.volume_fraction(img, phase=PHASE_ID)
    print(f"Volume Fraction: {vf.fraction:.6f}")

    # Percolation check
    for direction in ["x", "y", "z"]:
        perc = oi.percolation_check(img, phase=PHASE_ID, direction=direction)
        print(f"Percolates in {direction.upper()}: {perc.percolates}")

        if perc.percolates:
            result = oi.tortuosity(
                img,
                phase=PHASE_ID,
                direction=direction,
                solver="flexgmres",
                max_grid_size=MAX_GRID_SIZE,
            )
            print(f"  Tau_{direction.upper()} = {result.tortuosity:.6f}")
            print(f"  Iterations: {result.iterations}, Residual: {result.residual_norm:.2e}")

    t_total = time.time() - t0
    print(f"Total wall time: {t_total:.2f} seconds")
'''

print("=== HPC Python Script ===")
print(SCRIPT)

## 3. Using the C++ Parallel Readers

For datasets that exceed available RAM (common with synchrotron data), OpenImpala provides C++ readers that perform distributed I/O:

| Reader | File Types | Key Benefit |
|--------|------------|-------------|
| `TiffReader` | `.tif`, `.tiff` | Reads multi-page TIFF stacks or TIFF sequences |
| `HDF5Reader` | `.h5`, `.hdf5` | Parallel HDF5 I/O (requires parallel HDF5 build) |
| `RawReader` | `.raw` | Flat binary files (specify dimensions and data type) |
| `DatReader` | `.dat` | Legacy binary format |

The `oi.read_image()` function auto-detects the format from the file extension:

```python
# TIFF (auto-detected)
reader, img = oi.read_image("scan.tif", threshold=0.5)

# HDF5 with custom dataset path
reader, img = oi.read_image("scan.hdf5", threshold=0.5,
                            hdf5_dataset="/exchange/data")

# RAW binary (must specify dimensions and type)
reader, img = oi.read_image("scan.raw", threshold=128,
                            raw_width=2000, raw_height=2000, raw_depth=2000)
```

The returned `img` (a `VoxelImage`) can be passed directly to `oi.volume_fraction()`, `oi.percolation_check()`, and `oi.tortuosity()` -- exactly the same API as a NumPy array.

## 4. SLURM Batch Script

Below is a ready-to-use SLURM submission script. Adjust the paths and resource requests for your specific cluster.

In [ ]:
SLURM_SCRIPT = '''
#!/bin/bash
#SBATCH --job-name=openimpala
#SBATCH --nodes=4
#SBATCH --ntasks-per-node=32
#SBATCH --time=02:00:00
#SBATCH --output=openimpala_%j.log

# Load required modules (adjust for your cluster)
module load python/3.10
module load openmpi/4.1

# Ensure one MPI rank per core (no OpenMP threading)
export OMP_NUM_THREADS=1

# Activate your virtual environment
source ~/venvs/openimpala/bin/activate

# Run the analysis script with 128 MPI ranks across 4 nodes
mpirun -np $SLURM_NTASKS python hpc_tortuosity.py

echo "Job finished at $(date)"
'''

print("=== SLURM Submission Script ===")
print(SLURM_SCRIPT)
print("Save as 'submit.sh' and run with: sbatch submit.sh")

## 5. Performance Tips

### Choosing `max_grid_size`

| Dataset Size | Recommended `max_grid_size` | Reasoning |
|-------------|---------------------------|----------|
| $100^3$ | 64--128 | Few ranks, minimize box overhead |
| $500^3$ | 32--64 | Balance boxes across 16--64 ranks |
| $1000^3$+ | 32 | Many boxes for good load distribution |

### Memory Estimation

Each MPI rank needs memory for:
- The local portion of the voxel grid (int32: 4 bytes/voxel)
- The HYPRE matrix (sparse, ~56 bytes/voxel for 7-pt stencil)
- The solution vector and work arrays (~16 bytes/voxel)

**Rule of thumb:** ~80 bytes per active voxel per rank. A $1000^3$ dataset with 50% porosity on 64 ranks needs approximately:
```
500M active voxels * 80 bytes / 64 ranks = ~625 MB per rank
```

### Using the Container on HPC

If your cluster does not have OpenImpala's dependencies installed, you can use the pre-built Apptainer/Singularity container:

```bash
# Download the container from GitHub Releases
wget https://github.com/BASE-Laboratory/OpenImpala/releases/download/vX.Y.Z/openimpala-vX.Y.Z.sif

# Run the C++ application directly
mpirun -np 128 apptainer exec \
    -B "$(pwd):$(pwd)" openimpala-vX.Y.Z.sif \
    /usr/local/bin/Diffusion ./inputs

# Or run a Python script inside the container
mpirun -np 128 apptainer exec \
    -B "$(pwd):$(pwd)" openimpala-vX.Y.Z.sif \
    python hpc_tortuosity.py
```

## 6. Quick Demo: Scaling with Grid Size

To illustrate the effect of `max_grid_size` on a laptop, let's time the solver on a synthetic structure with different decomposition settings.

In [ ]:
!pip install -q openimpala matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi

# Generate a 64^3 test structure
np.random.seed(42)
N = 64
data = np.random.choice([0, 1], size=(N, N, N), p=[0.4, 0.6]).astype(np.int32)

grid_sizes = [16, 32, 64]
timings = []

with oi.Session():
    for gs in grid_sizes:
        t0 = time.time()
        try:
            result = oi.tortuosity(data, phase=1, direction="z",
                                   solver="flexgmres", max_grid_size=gs)
            t_elapsed = time.time() - t0
            timings.append(t_elapsed)
            print(f"max_grid_size={gs:3d}: Tau={result.tortuosity:.4f}, "
                  f"Time={t_elapsed:.3f}s, Iterations={result.iterations}")
        except Exception as e:
            timings.append(None)
            print(f"max_grid_size={gs:3d}: Failed ({e})")

# Plot the timing comparison
valid = [(gs, t) for gs, t in zip(grid_sizes, timings) if t is not None]
if valid:
    gs_vals, t_vals = zip(*valid)
    fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
    ax.bar([str(g) for g in gs_vals], t_vals, color='#1f77b4')
    ax.set_xlabel("max_grid_size", fontweight='bold')
    ax.set_ylabel("Wall Time (seconds)", fontweight='bold')
    ax.set_title(f"Domain Decomposition Timing ({N}^3 grid, 1 rank)", fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

## Summary

| Scenario | Approach |
|----------|----------|
| **Laptop / Colab** | `pip install openimpala`, use NumPy arrays directly |
| **Small cluster (1–16 cores)** | `mpirun -np 16 python script.py` with NumPy loading |
| **Large cluster (16–128+ cores)** | `mpirun -np 128 python script.py` with `oi.read_image()` for parallel I/O |
| **HPC without Python** | `mpirun -np 128 Diffusion ./inputs` (pure C++ application) |

The API is the same at every scale — only the launch command changes.

**Back to the beginning:**
- [Tutorial 1: Getting Started](01_hello_openimpala.ipynb) — Core workflow with synthetic microstructures.
- [Tutorial 2: From 3D Image to Device Model](02_digital_twin.ipynb) — Load real CT scans and export to PyBaMM.

---

## References & Further Reading

1. **OpenImpala:** S. Mayner, F. Ciucci, *OpenImpala: open-source computational framework for microstructural analysis of 3D tomography data*, SoftwareX (2024). [GitHub](https://github.com/BASE-Laboratory/OpenImpala)
2. **AMReX:** W. Zhang et al., *AMReX: a framework for block-structured adaptive mesh refinement*, J. Open Source Software 4(37), 1370 (2019). [doi:10.21105/joss.01370](https://doi.org/10.21105/joss.01370)
3. **AMReX scaling:** A. S. Almgren et al., *Block-structured adaptive mesh refinement — theory, implementation and application*, J. Comput. Physics 332, 1–28 (2017). [doi:10.1016/j.jcp.2016.12.073](https://doi.org/10.1016/j.jcp.2016.12.073)
4. **HYPRE:** R. D. Falgout & U. M. Yang, *hypre: A library of high performance preconditioners*, Computational Science — ICCS 2002, LNCS 2331, pp. 632–641 (2002). [doi:10.1007/3-540-47789-6_66](https://doi.org/10.1007/3-540-47789-6_66)
5. **Parallel HDF5:** The HDF Group, *HDF5 — Parallel I/O*, [hdfgroup.org](https://www.hdfgroup.org/solutions/hdf5/)
6. **Apptainer/Singularity for HPC:** G. M. Kurtzer et al., *Singularity: Scientific containers for mobility of compute*, PLoS ONE 12(5), e0177459 (2017). [doi:10.1371/journal.pone.0177459](https://doi.org/10.1371/journal.pone.0177459)
7. **MPI standard:** Message Passing Interface Forum, *MPI: A Message-Passing Interface Standard, Version 4.0* (2021). [mpi-forum.org](https://www.mpi-forum.org/docs/mpi-4.0/mpi40-report.pdf)